<a href="https://colab.research.google.com/github/GopalKrishna-India/Geospatial/blob/master/KoboSurveyDataDownload/DownloadData%26Media_from_KobotoolBox.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [7]:
import os
import io
import requests
import json
import pandas as pd
from PIL import Image
from concurrent.futures import ThreadPoolExecutor, as_completed
from tqdm.notebook import tqdm

# ==============================================================================
# 1. CONFIGURATION SETTINGS
# ==============================================================================
KOBO_URL = "https://kf.kobotoolbox.org"
API_TOKEN = "101508eb61f5625546c866c3dfbb55b5fec8fe67"
ASSET_UID = "arDwQ2JShtAZzV9GtrfneU"

# Output storage parameters
MEDIA_DIR = "/content/drive/MyDrive/Kobo-SpecLibFieldData/kobo_media_files"
OUTPUT_EXCEL = "/content/drive/MyDrive/Kobo-SpecLibFieldData/kobo_survey_data_local_media.xlsx"

# Download & Compression parameters
MAX_WORKERS = 10         # Parallel workers
JPEG_QUALITY = 75        # Quality setting (75 gives ~30-40% file size reduction without visual detail loss)

HEADERS = {"Authorization": f"Token {API_TOKEN}"}
os.makedirs(MEDIA_DIR, exist_ok=True)

# ==============================================================================
# 2. FETCH ALL SUBMISSION RECORDS (PAGINATED)
# ==============================================================================
print("Fetching submission data from KoboToolbox API...")
data_url = f"{KOBO_URL}/api/v2/assets/{ASSET_UID}/data.json?page_size=1000"

results = []
while data_url:
    response = requests.get(data_url, headers=HEADERS)
    if response.status_code != 200:
        raise Exception(f"Failed to fetch data: {response.status_code} - {response.text}")

    data_json = response.json()
    batch = data_json.get("results", [])
    results.extend(batch)
    data_url = data_json.get("next")

print(f"Successfully retrieved all {len(results)} submission records!\n")

# ==============================================================================
# 3. EXTRACT FULL-RESOLUTION MEDIA QUEUE
# ==============================================================================
media_tasks = []
for record in results:
    attachments = record.get("_attachments", [])
    for attachment in attachments:
        download_url = attachment.get("download_url")
        filename = attachment.get("filename", "").split("/")[-1]

        if download_url and filename:
            local_filepath = os.path.join(MEDIA_DIR, filename)
            if not os.path.exists(local_filepath):
                media_tasks.append((download_url, local_filepath))

print(f"Found {len(media_tasks)} media files to process...")

# ==============================================================================
# 4. DOWNLOAD & COMPRESS IMAGES (30% SIZE REDUCTION)
# ==============================================================================
def download_and_compress(task):
    url, filepath = task
    try:
        res = requests.get(url, headers=HEADERS, stream=True, timeout=30)
        if res.status_code == 200:
            # Load raw image into memory
            img_data = res.content

            # Compress image if it's a supported format (JPEG / PNG)
            try:
                img = Image.open(io.BytesIO(img_data))
                if img.mode != 'RGB':
                    img = img.convert('RGB')

                # Save compressed image (~30-40% smaller)
                img.save(filepath, format="JPEG", quality=JPEG_QUALITY, optimize=True)
            except Exception:
                # Fallback: Write original binary if format is unreadable by Pillow
                with open(filepath, "wb") as f:
                    f.write(img_data)
            return True
    except Exception:
        return False
    return False

# Execute parallel compressed downloads
if media_tasks:
    with ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
        futures = [executor.submit(download_and_compress, task) for task in media_tasks]

        for _ in tqdm(as_completed(futures), total=len(futures), desc="Downloading & Compressing Media", unit="file"):
            pass
else:
    print("All media files are already downloaded locally.")

# ==============================================================================
# 5. RELINK FILE PATHS & EXPORT TO EXCEL
# ==============================================================================
processed_records = []
for record in results:
    record_dict = dict(record)
    attachments = record_dict.get("_attachments", [])
    for attachment in attachments:
        filename = attachment.get("filename", "").split("/")[-1]
        if filename:
            local_filepath = os.path.join(MEDIA_DIR, filename)
            for key, val in list(record_dict.items()):
                if isinstance(val, str) and filename in val:
                    record_dict[key] = local_filepath
    processed_records.append(record_dict)

df = pd.DataFrame(processed_records)
if "_attachments" in df.columns:
    df.drop(columns=["_attachments"], inplace=True)

df.to_excel(OUTPUT_EXCEL, index=False)
print(f"\nProcessing Complete!")
print(f"1. Compressed photos saved to: '{os.path.abspath(MEDIA_DIR)}'")
print(f"2. Updated Excel dataset saved to: '{os.path.abspath(OUTPUT_EXCEL)}'")

Fetching submission data from KoboToolbox API...
Successfully retrieved all 632 submission records!

Found 1892 media files to process...


IsADirectoryError: [Errno 21] Is a directory: '/content/drive/MyDrive/Kobo-SpecLibFieldData/kobo_survey_data_local_media.xlsx'

In [15]:
## Link photos with the Google Sheets (Excel file); so that photos can be viewed directly from Excel; Example:Column "AN Column"-Link-https://docs.google.com/spreadsheets/d/16CgSb2nTM5UXLTQukZa7B2KzgRGC_iHM/edit?gid=823073299#gid=823073299##
import os
import pandas as pd
from google.colab import drive, auth
from googleapiclient.discovery import build

# # 1. Mount Google Drive
# drive.mount('/content/drive')

# 2. Authenticate to get direct Google Drive File IDs
auth.authenticate_user()
drive_service = build('drive', 'v3')

# Set your exact Excel path and target root media folder path
excel_path = '/content/drive/MyDrive/Kobo-SpecLibFieldData/kobo_survey_data_local_media.xlsx'  # CHANGE TO YOUR ACTUAL EXCEL FILE PATH
media_folder_path = '/content/drive/MyDrive/Kobo-SpecLibFieldData/kobo_media_files'

print("Indexing files in your Google Drive folder...")

# 3. Recursively map all image filenames to their Google Drive File IDs
file_map = {}

# Query Drive API recursively for all files
query = "trashed = false and mimeType != 'application/vnd.google-apps.folder'"
page_token = None

while True:
    response = drive_service.files().list(
        q=query,
        fields="nextPageToken, files(id, name)",
        pageToken=page_token,
        pageSize=1000
    ).execute()

    for f in response.get('files', []):
        file_map[f['name']] = f['id']

    page_token = response.get('nextPageToken', None)
    if not page_token:
        break

print(f"Successfully indexed {len(file_map)} files across Drive.")

# 4. Helper function to extract exact filename and convert to HYPERLINK
def clean_path_to_hyperlink(raw_path):
    if pd.isna(raw_path) or not str(raw_path).strip():
        return ""

    # Extract just the filename (e.g., "1790065218627.jpg") from full paths
    clean_filename = os.path.basename(str(raw_path).strip())

    # Check if filename exists in our Drive index
    file_id = file_map.get(clean_filename)

    if file_id:
        return f'=HYPERLINK("https://drive.google.com/file/d/{file_id}/view", "View Photo")'
    else:
        return "File Not Found"

# 5. Process the Excel file
df = pd.read_excel(excel_path)

photo_columns = [
    'sec_photos/photo_full_tree',
    'sec_photos/photo_crown_sky',
    'sec_photos/photo_surrounding_site'
]

for col in photo_columns:
    if col in df.columns:
        df[col] = df[col].apply(clean_path_to_hyperlink)

# 6. Save output
output_path = '/content/drive/MyDrive/Kobo-SpecLibFieldData/kobo_data_with_links.xlsx'
df.to_excel(output_path, index=False)
print("Done! File saved to:", output_path)

Indexing files in your Google Drive folder...
Successfully indexed 6519 files across Drive.
Done! File saved to: /content/drive/MyDrive/Kobo-SpecLibFieldData/kobo_data_with_links.xlsx
